# Generate description embeddings
This notebook trains a text encoder using a contrastive objective to encode descriptions into embeddings that would be linked to the associated fixed embeddings of the image objects extracted from the cross-modal decoder of Grounding DINO.

### 0. Import libraries and load data

In [ ]:
import os
import re
import copy
import json
import shutil

import torch
import polars as pl
from tqdm import tqdm
import plotly.express as px
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

PROCESSED_DATA_PATH = "../../data/processed/"
INPUT_PATH_EMBEDDINGS = "../../../Open-Grounding-DINO/embeddings_data/"
EMBEDDINGS_FILE_NAME = "fine_tuned_embeddings_{}_trained.json"
STORAGE_PATH = "../../experiments/contrastive_training/"
COLORS = ["#cd968e", "#acb0e0", "#aecbdc", "#bcd5c3", "#bfbfbf"]

In [ ]:
def load_paintings_objects():
    with open(f"{PROCESSED_DATA_PATH}paintings_with_filtered_objects.json") as f:
        all_annotations = json.load(f)

    object_labels = []
    object_descriptions = []

    for annotation in all_annotations:
        object_labels.append(list(annotation["objects"].keys()))
        object_descriptions.append(
            [obj_data["description"] for obj_data in annotation["objects"].values()]
        )

    paintings_objects = copy.deepcopy(all_annotations)

    for index, painting_objects in enumerate(paintings_objects):
        paintings_objects[index]["objects"] = object_labels[index]
        paintings_objects[index]["object_description"] = object_descriptions[index]

    paintings_objects = (
        pl.from_dicts(paintings_objects)
        .explode("objects", "object_description")
        .with_columns((pl.col("id").cast(pl.String) + "_" + pl.col("objects")).alias("id_join"))
    )
    
    return paintings_objects

In [ ]:
def load_descriptions_and_image_embeddings(set_name, paintings_objects, keep_unique_descriptions=False):
    # # leave out this paintings as their descriptions are too long
    leave_out_painting_ids = [
        117,
        133,
        134,
        168,
        195,
        257,
        258,
        260,
        266,
        278,
        297,
        320,
        336,
        390,
        438,
        440,
        488,
        506,
        606,
        641,
        658,
        663,
        667,
        675,
        695,
        720,
        734,
        747,
        772,
        802,
        806,
        840,
        852,
        869,
        880,
        881,
        922,
        962,
        963,
        1086,
        1089,
        1117,
        1130,
        1131,
        1132,
        1151,
        1273,
        1289,
        1319,
        1376,
        1379,
        1434,
        1477,
        1483,
        1494,
        1508,
        1525,
        1533,
        1585,
        1650,
        2004,
        2146,
        2161,
        2744,
        2755,
        2929,
        3280,
        3538,
        3877,
        4085,
        4381,
        4909,
        5048,
        5525,
        5969,
        6308,
        7476,
        7574,
        7917,
        8402,
        8423,
        8627,
        8671,
        9211,
        9257,
        10037,
        11093,
        11298,
        11348,
        11353,
        11407,
        11671,
        12014,
        101,
        459,
        614,
        770,
        791,
        805,
        938,
        988,
        2521,
        3139,
        3530,
        9090,
        9584,
        10779,
        662,
        743,
        848,
        1263,
        1724,
        4857,
        5529,
        7550,
        9177,
        9645,
        10071,
    ]

    embeddings_data = (
        pl.read_json(f"{INPUT_PATH_EMBEDDINGS}{EMBEDDINGS_FILE_NAME.format(set_name)}")
        .explode(pl.all())
        .with_columns((pl.col("painting_id").cast(pl.String) + "_" + pl.col("text")).alias("id_join"))
    )

    embeddings_data = (
        embeddings_data.join(paintings_objects, on="id_join", how="left")
        .drop("text")
        .rename({"object_description": "text"})
        .select(*embeddings_data.columns)
        .drop("id_join", "text_embedding_backbone", "text_embedding_enhanced")
        .with_columns(
            pl.col("text")
            .map_elements(
                lambda text: re.sub(r" +", " ", text.replace(".", " | ").strip().lower()),
                return_dtype=pl.String,
            )
            .alias("text")
        ).filter(~pl.col("painting_id").is_in(leave_out_painting_ids))
    )

    if keep_unique_descriptions:
        embeddings_data = (
            embeddings_data.sort("probability", descending=True)
            .group_by("text", maintain_order=True)
            .agg(pl.all().first())
            .sort("painting_id")
        )
        
    return embeddings_data


In [ ]:
paintings_objects = load_paintings_objects()

In [ ]:
embeddings_data_train = load_descriptions_and_image_embeddings(
    "train", paintings_objects, keep_unique_descriptions=True
)
embeddings_data_train

In [ ]:
embeddings_data_val = load_descriptions_and_image_embeddings(
    "val", paintings_objects, keep_unique_descriptions=True
)
embeddings_data_val

In [ ]:
embeddings_data_test = load_descriptions_and_image_embeddings(
    "test", paintings_objects, keep_unique_descriptions=True
)
embeddings_data_test

### 1. Define PyTorch data loaders

In [ ]:
class CrossModalDataset(Dataset):
    def __init__(self, descriptions, image_embeddings, tokenizer, max_length):
        self.descriptions = descriptions
        self.image_embeddings = image_embeddings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        description = self.descriptions[idx]
        image_embedding = self.image_embeddings[idx]

        encoding = self.tokenizer(
            description,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        preprocessed_data = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image_embedding": image_embedding,
        }

        return preprocessed_data

### 2. Define text encoder with projector

In [ ]:
class TextEncoder(torch.nn.Module):
    def __init__(self, transformer, tokenizer, pooling, output_dim=256, dropout=0.2):
        super().__init__()
        self.transformer = transformer
        self.tokenizer = tokenizer
        self.pooling = pooling
        self.output_dim = output_dim
        self.dropout = dropout
        self.transformer_output_dim = self.transformer.config.hidden_size

        
        self.projector = torch.nn.Sequential(
            torch.nn.Linear(self.transformer_output_dim, 512),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),                    
            torch.nn.Linear(512, self.output_dim),         
            torch.nn.LayerNorm(self.output_dim)            
        )

    def __mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output["last_hidden_state"]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
            input_mask_expanded.sum(1), min=1e-9
        )

    def __cls_pooling(self, model_output):
        return model_output.last_hidden_state[:, 0]

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        if self.pooling == "mean":
            pooled_output = self.__mean_pooling(outputs, attention_mask)
        elif self.pooling == "cls":
            pooled_output = self.__cls_pooling(outputs)
        else:
            raise ValueError("Undefined pooling method.")

        projected_embeddings = self.projector(pooled_output)

        normalized_projected_embeddings = torch.nn.functional.normalize(
            projected_embeddings, p=2, dim=1
        ).double()

        return normalized_projected_embeddings

    def encode_text(self, texts, batch_size=32, max_length=128):
        self.eval()
        all_embeddings = []

        with torch.no_grad():
            for i in tqdm(range(0, len(texts), batch_size)):
                batch_texts = texts[i : i + batch_size]

                encoding = self.tokenizer(
                    batch_texts,
                    truncation=True,
                    padding="max_length",
                    max_length=max_length,
                    return_tensors="pt",
                )

                input_ids = encoding["input_ids"].to(next(self.parameters()).device)
                attention_mask = encoding["attention_mask"].to(next(self.parameters()).device)

                embeddings = self.forward(input_ids, attention_mask)
                all_embeddings.append(embeddings.cpu())

        return torch.cat(all_embeddings, dim=0)

### 3. Define loss function and computation

In [ ]:
def contrastive_loss(text_embeddings, image_embeddings, temperature):
    text_embeddings = torch.nn.functional.normalize(text_embeddings, dim=-1)

    logits = torch.matmul(text_embeddings, image_embeddings.T) / temperature
    labels = torch.arange(len(text_embeddings)).to(logits.device)

    loss = torch.nn.functional.cross_entropy(logits, labels)

    return loss

In [ ]:
def compute_loss(text_encoder, data_loader, temperature, device):
    text_encoder.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            image_embeddings = batch["image_embedding"].to(device)

            text_embeddings = text_encoder(input_ids, attention_mask)
            loss = contrastive_loss(text_embeddings, image_embeddings, temperature)

            total_loss += loss.item()

    return total_loss / len(data_loader)

### 4. Train

#### 4.1. Define hyperparameters, load data and perform the initialization

In [ ]:
device = "cuda:1"
experiment_name = "mean_pooling_frozen_encoder_dif_lr"
max_length = 256
num_epochs = 30
batch_size = 64
learning_rate = 5e-5
weight_decay = 0.01
pooling = "mean"
dropout = 0.2
temperature = 0.1
text_encoder_name = "sentence-transformers/all-mpnet-base-v2"

In [ ]:
try:
    shutil.rmtree(f"{STORAGE_PATH}{experiment_name}")
except:
    pass

os.mkdir(f"{STORAGE_PATH}{experiment_name}")

In [ ]:
# load data and create batches
tokenizer = AutoTokenizer.from_pretrained(text_encoder_name)

train_data = CrossModalDataset(
    embeddings_data_train["text"].to_numpy(),
    embeddings_data_train["embedding_object_image"].to_numpy(),
    tokenizer,
    max_length,
)
val_data = CrossModalDataset(
    embeddings_data_val["text"].to_numpy(),
    embeddings_data_val["embedding_object_image"].to_numpy(),
    tokenizer,
    max_length,
)
test_data = CrossModalDataset(
    embeddings_data_test["text"].to_numpy(),
    embeddings_data_test["embedding_object_image"].to_numpy(),
    tokenizer,
    max_length,
)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

# initialize the text encoder which includes the projector
transformer = AutoModel.from_pretrained(text_encoder_name)
text_encoder = TextEncoder(transformer, tokenizer, pooling=pooling, dropout=dropout).to(device)

# define the optimizer and the scheduler for decaying the learning rate
# optimizer = torch.optim.AdamW(
#     text_encoder.parameters(), lr=learning_rate, weight_decay=weight_decay
# )


optimizer = torch.optim.AdamW([
    {'params': text_encoder.transformer.parameters(), 'lr': learning_rate / 10},
    {'params': text_encoder.projector.parameters(), 'lr': learning_rate},
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

#### 4.2. Run the training loop and store results

In [ ]:
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    text_encoder.train()
    epoch_train_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image_embeddings = batch["image_embedding"].to(device)

        text_embeddings = text_encoder(input_ids, attention_mask)
        loss = contrastive_loss(text_embeddings, image_embeddings, temperature)

        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()

    scheduler.step()
    train_losses.append(epoch_train_loss / len(train_loader))
    val_losses.append(compute_loss(text_encoder, val_loader, temperature, device))

    print(f"Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}")

test_loss = compute_loss(text_encoder, test_loader, temperature, device)
print(f"Test loss: {test_loss:.4f}")

In [ ]:
losses = {
    "epoch": list(range(1, num_epochs + 1)),
    "train_losses": train_losses,
    "val_losses": val_losses,
}
losses_df = pl.from_dict(losses)

fig = px.line(
    losses_df.rename({"train_losses": "train", "val_losses": "val"}),
    x="epoch",
    y=["train", "val"],
    labels={"value": "NT-Xnet loss value"},
    title="Evolution of the contrastive loss value",
    markers=True,
    color_discrete_sequence=COLORS,
)

fig.update_layout(width=1400, height=450)

fig.show()
import plotly.io as pio
pio.write_html(fig, file=f"{STORAGE_PATH}{experiment_name}/loss_evolution.html", auto_open=False)
#fig.write_image(f"{STORAGE_PATH}{experiment_name}/loss_evolution.png", scale=3)

In [ ]:
losses["test_loss"] = test_loss
del losses["epoch"]
results = {
    "max_length": max_length,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "pooling": pooling,
    "temperature": temperature,
    "text_encoder_name": text_encoder_name,
    "losses": losses,
}
with open(f"{STORAGE_PATH}{experiment_name}/results.json", "w") as f:
    json.dump(results, f, indent=4)

torch.save(text_encoder.state_dict(), f"{STORAGE_PATH}{experiment_name}/encoder.pth")

### 5. Generate embeddings for descriptions

In [ ]:
text_encoder = TextEncoder(transformer, tokenizer, pooling=pooling, dropout=dropout).to(device)
text_encoder.load_state_dict(torch.load(f"{STORAGE_PATH}{experiment_name}/encoder.pth"))

In [ ]:
paintings_objects = load_paintings_objects()
inference_data = load_descriptions_and_image_embeddings(
    "test", paintings_objects, keep_unique_descriptions=False
)
inference_data

In [ ]:
embeddings = text_encoder.encode_text(inference_data["text"].to_list(), batch_size, max_length)
embeddings_list = [list([float(val) for val in embedding]) for embedding in embeddings.cpu().numpy()]
inference_data_updated = inference_data.with_columns(pl.Series("text_embedding_enhanced", embeddings_list)).to_dicts()

In [ ]:
with open(f"{INPUT_PATH_EMBEDDINGS}fine_tuned_embeddings_test_{experiment_name}.json", "w") as f:
    json.dump(inference_data_updated, f)

### 6. Analyze similarities and performance

In [ ]:
train_loader = DataLoader(train_data, batch_size=len(train_data), shuffle=False)
val_loader = DataLoader(val_data, batch_size=len(val_data), shuffle=False)
test_loader = DataLoader(test_data, batch_size=len(test_data), shuffle=False)

In [ ]:
def evaluate_retrieval(text_encoder, dataloader, device, k_values=[1, 5, 10]):
    """
    Evaluate retrieval performance
    """
    text_encoder.eval()
    all_text_embeddings = []
    all_image_embeddings = []
    positive_similarities = []
    negative_similarities = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            image_embeddings = batch['image_embedding'].to(device)
            
            text_embeddings = text_encoder(input_ids, attention_mask)
            
            all_text_embeddings.append(text_embeddings.cpu())
            all_image_embeddings.append(image_embeddings.cpu())
            
            similarities = torch.matmul(text_embeddings, image_embeddings.T)

            positive_similarities.extend(torch.diag(similarities).cpu().numpy())

            neg_similarities_matrix = similarities.clone()
            mask_positive = torch.eye(similarities.size(0), dtype=torch.bool, device=device)
            neg_similarities_matrix.masked_fill_(mask_positive, -torch.inf)
            negative_similarities.extend(neg_similarities_matrix.max(dim=1).values.cpu().numpy())
    
    all_text_embeddings = torch.cat(all_text_embeddings, dim=0)
    all_image_embeddings = torch.cat(all_image_embeddings, dim=0)
    
    similarity_matrix = torch.matmul(all_text_embeddings, all_image_embeddings.T)
    
    recalls = {}
    for k in k_values:
        _, top_k_indices = torch.topk(similarity_matrix, k, dim=1)
        
        correct_indices = torch.arange(len(all_text_embeddings)).unsqueeze(1)
        recall_at_k = (top_k_indices == correct_indices).any(dim=1).float().mean().item()
        recalls[f'Recall@{k}'] = recall_at_k
    
    return recalls, positive_similarities, negative_similarities

In [ ]:
recalls, positive_similarities, negative_similarities = evaluate_retrieval(text_encoder, test_loader, device, k_values=[1, 5, 10])
recalls

In [ ]:
recalls, positive_similarities, negative_similarities = evaluate_retrieval(text_encoder, val_loader, device, k_values=[1, 5, 10])
recalls

In [ ]:
recalls, positive_similarities, negative_similarities = evaluate_retrieval(text_encoder, train_loader, device, k_values=[1, 5, 10])
recalls

In [ ]:
for i in range(20):
    print(positive_similarities[i], negative_similarities[i])